# KronLLM S10 — supervised fine-tuning

Starts from the Q10 decay-pass weights (weights only, fresh AdamW) and trains
4,800 steps x 16,384 tokens = 78.6M tokens on the measured SFT corpus: 513M
tokens of math (with the column drills), code, open Q&A, forum, chat and
identity, 60/40 EN/RU by bytes.

The prompt is not data here.  `model.mask_user_turns` is on, so everything up to
and including `<|assistant|>` is masked and only the answer is a target --
`prompt_tokens` in `sft_index.npy` marks exactly that span.

4,800 rather than 4,000 steps is a quota decision, not a recipe: the decay pass
ended with about an hour of the week's GPU hours left, and a shorter pass would
have idled on the wall instead of training.

Learning rate 5e-5 -> 5e-6 over the 4,800 steps, 100 warmup steps.  Checkpoints
every 250 steps to `/kaggle/working/s10`.


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

print(sorted(os.listdir('/kaggle/input')))
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
for line in open('/proc/meminfo'):
    if line.startswith(('MemTotal', 'MemAvailable')):
        print(line.strip())
try:
    print('cgroup max', open('/sys/fs/cgroup/memory.max').read().strip())
except OSError:
    pass
print('working disk', shutil.disk_usage('/kaggle/working').free // 2**30, 'GiB free')

# Dataset mounts moved to /kaggle/input/datasets/<owner>/<slug> on newer
# images; support both layouts.
def find_dataset(slug, probe):
    roots = glob.glob(f'/kaggle/input/datasets/*/{slug}') + [f'/kaggle/input/{slug}']
    for root in roots:
        if os.path.exists(os.path.join(root, probe)):
            return root
    raise FileNotFoundError(f'{slug}: {probe} not found under /kaggle/input')

SRC = find_dataset('kron-sft-corpus', 'train.bpe.bin')
def find_code():
    # The code dataset is versioned (kron-llm-code-vNN) and the version
    # changes with the code, so find the mounted package instead of
    # hard-coding a slug that goes stale on the next bump.
    hits = sorted(glob.glob('/kaggle/input/**/kronllm/__init__.py',
                            recursive=True))
    if not hits:
        raise FileNotFoundError('kronllm package not found under /kaggle/input')
    return os.path.dirname(os.path.dirname(hits[0]))

CODE = find_code()
print('SRC', SRC)
print('CODE', CODE)
sys.path.insert(0, CODE)

# The bins live on the dataset mount; a disk-backed copy keeps two
# DDP ranks under the 32 GB cgroup (tmpfs pages are charged twice).
# /kaggle/working, which is saved as kernel output and would upload 4 GB).
# through that mount were OOM-killed; a local copy is cheap and predictable.
DATA = '/kaggle/working/data'
os.makedirs(DATA, exist_ok=True)
for name in ('train.bpe.bin', 'val.bpe.bin', 'tokenizer.json', 'tokenizer_meta.json'):
    src = os.path.join(SRC, name)
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print('copying', name, round(os.path.getsize(src) / 1e9, 2), 'GB')
        shutil.copyfile(src, dst)
# The val bucket index ships with the code dataset (it describes the val
# bin but is tiny), so the slice-aware validation can label every block.
idx_src = os.path.join(CODE, 'val_index.json')
if os.path.exists(idx_src):
    shutil.copyfile(idx_src, os.path.join(DATA, 'val_index.json'))
    print('val index', os.path.getsize(os.path.join(DATA, 'val_index.json')))
else:
    print('no val_index.json in the code dataset: slice eval will fall back')

print('local bins', {p: round(os.path.getsize(os.path.join(DATA, p)) / 1e9, 2)
                     for p in ('train.bpe.bin', 'val.bpe.bin')})

import kronllm
print('kronllm', kronllm.__file__)

import json as _json
_meta_path = os.path.join(DATA, 'tokenizer_meta.json')
_meta = _json.load(open(_meta_path, encoding='utf-8'))
_meta['tokenizer'] = os.path.join(DATA, 'tokenizer.json')
with open(_meta_path, 'w', encoding='utf-8') as _fh:
    _json.dump(_meta, _fh, indent=1)
print('tokenizer ->', _meta['tokenizer'])

In [ ]:
import json
meta = json.load(open(DATA + '/tokenizer_meta.json'))
print('vocab', meta['vocab_size'], 'chat ids', meta.get('chat_tokens'))
print('train tokens', meta['train_tokens'], 'val tokens', meta['val_tokens'])
print('bytes/token', meta['bytes_per_token'])
print('token share', meta.get('token_share'), 'byte share', meta.get('byte_share'))

In [ ]:
import torch

# DDP is unstable in this image: two ranks each map the 4.3 GB bin and the
# 32 GB cgroup kills one of them at the first collective.  One T4 with fp16
# and batch 32 is the measured configuration (6.5k tokens/s, 15.2 GB VRAM).
n_gpu = 1
print('GPUs available:', torch.cuda.device_count(), '| using', n_gpu)

# The session stops itself before Kaggle's 500-minute kill: the budget
# writes last.pt on the way out, so nothing is lost to the wall clock.
MINUTES = 205   # 4,800 steps ~ 3.4 h; the platform cap is 720 min
                # copying the 1.1 GB SFT bin costs ~2 min, so 690 left no room
                # for a timed-out kernel to still publish its output.
cmd = [sys.executable, '-u', '-m', 'kronllm.cli', 'train', '--exp', 'S10',
       '--minutes', str(MINUTES),
       '--data-dir', DATA, '--run-dir', '/kaggle/working/s10',
       '--override', 'training.log_every=50',
       # 4,800 x 16,384 = 78.6M tokens, 15.6 % of the SFT reservoir: the
       # longest pass that still leaves the epoch -- which memorises at
       # this size -- comfortably out of reach.
       '--override', 'training.max_steps=4800',
       '--override', 'training.schedule_steps=4800',
       # The corpus was rebuilt (new domains, new sizes), so the trainer
       # would refuse to resume on a different data_key without this.
       '--override', 'training.allow_data_change=true']

init = sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True))
if not init:
    raise FileNotFoundError('/kaggle/input has no last.pt: S10 starts from the '
                            'decay pass, not from scratch')
# A stale dataset version is invisible in the training log and only shows up
# as a mysteriously worse model, so the step is checked instead of trusted.
_probe = torch.load(init[0], map_location='cpu', weights_only=False)
_step = int(_probe.get('step') or 0)
print('initialising weights from', init[0], '| step', _step)
del _probe
if _step < 108507:
    raise SystemExit(
        f'the mounted checkpoint is at step {_step:,}, the decay ends at '
        f'109,507 (>=108,507 accepted): this is a stale dataset version -- '
        f're-push S10 once kron-q10-ckpt shows the new version')
cmd += ['--override', 'training.init_from=' + init[0]]
print(' '.join(cmd))
env = dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED='1',
           CUDA_VISIBLE_DEVICES='0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
try:
    subprocess.run(cmd, check=True, env=env)
finally:
    # Drop the 4.3 GB local copy so it is not uploaded as output.
    shutil.rmtree(DATA, ignore_errors=True)


In [ ]:
print(open('/kaggle/working/s10/results.json').read()[:1200])
for name in ('best.pt', 'last.pt', 'log.csv'):
    p = '/kaggle/working/q10/' + name
    if os.path.exists(p):
        print(name, round(os.path.getsize(p) / 1e6, 1), 'MB')
print('download last.pt and upload it as a dataset to resume next session')